# Tez İlerleme Sunumu

**Tez konusu.** Kubernetes ortamlarında ağ telemetrisi ve çalışma zamanı olaylarıyla
çok katmanlı anomali tespiti.

**Özet.** Tez, bir Kubernetes kümesindeki iki telemetri katmanını (ağ ve çalışma
zamanı) tek bir modelde birleştirmeyi öneriyor. Topladığımız ilk veri setinde iki sorun
çıktı. Birincisi hacim: model eğitmeye yetecek veri yok. İkincisi daha ciddi: çalışma
zamanı katmanı, "normal" davranış hakkında neredeyse hiçbir şey söylemiyor. İkincisi
doğrudan tezin özgün katkısını ilgilendiriyor, çünkü katkı tam olarak o katmanı
kullanmaya dayanıyor. Şu anda veriyi yeniden topluyoruz.

| # | Bölüm | Ne anlatıyor |
|---|---|---|
| 1 | Tez ne yapıyor | Problem, özgün katkı ve araştırma sorusu |
| 2 | Veri nasıl toplanıyor | Window, feature, label, senaryo |
| 3 | Neredeyiz | Şu ana kadar yapılanlar ve yol haritası |
| 4 | Bulgu 1 | Veri hacmi yetmiyor |
| 5 | Bulgu 2 | Çalışma zamanı katmanı normalde sessiz |
| 6 | Bulgu 2'nin kanıtı | Tek kurallı deneme |
| 7 | Neden ilk bakışta görünmedi | Ayrılabilirlik skorunun paydası |
| 8 | Plan | Veriyi nasıl yeniden topluyoruz |
| 9 | Şu anki durum | Kabul kriterleri |
| 10 | Sıradaki adım | Ve riskler |
| — | Sözlük | Terimlerin toplu açıklaması |

## Altyapı

Veri dosyasını yükler ve güncel olup olmadığını kontrol eder.

In [1]:
import sys
from pathlib import Path

# `style.py` bu notebook'un yanında (notebooks/). Kernel çalışma dizini normalde
# notebook'un kendi dizinidir; JupyterLab'in kökü repo kökü verilmişse diye tek adım
# aşağı da denenir.
HERE = Path.cwd().resolve()
if not (HERE / "style.py").is_file():
    HERE = HERE / "notebooks"
sys.path.insert(0, str(HERE))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import style
import charts

REPO = style.bootstrap()      # repo kökü + src/ import yolunda
style.apply_theme()

# Ölçüm mantığının tek kaynağı — bu notebook'ta hiçbir ölçüm yeniden yazılmıyor:
import validate_dataset as vd
from features.extract_features import ALL_COLS, NET_COLS, RT_COLS

FEATURES = REPO / "src/data/features/features.parquet"
RAW_DIR = REPO / "src/data/raw"
WINDOW = "10s"

df = pd.read_parquet(FEATURES)

# Tek satırlık tazelik kontrolü: veri dosyası ham toplamadan sonra üretilmiş mi?
parts = sorted(RAW_DIR.glob("*.part*.parquet"))
yeni_part = max(parts, key=lambda p: p.stat().st_mtime) if parts else None
features_zamani = pd.Timestamp(FEATURES.stat().st_mtime, unit="s", tz="UTC")
stale = bool(yeni_part) and yeni_part.stat().st_mtime > FEATURES.stat().st_mtime

print(f"veri dosyası : features.parquet · {features_zamani:%Y-%m-%d} · {len(df)} satır")
print("UYARI: VERİ ESKİMİŞ — ölçümler eski bir toplama koşusundan geliyor."
      if stale else "veri dosyası ham toplamayla tutarlı")

summary = vd.summarize(df)
coverage = vd.normal_runtime_coverage(df)

veri dosyası : features.parquet · 2026-10-11 · 101 satır
UYARI: VERİ STALE — ölçümler eski bir toplama koşusuna ait.


## 1 · Tez ne yapıyor?

### İzlenen sistem

Kubernetes, bir uygulamayı çok sayıda küçük servise bölüp bir sunucu kümesi üzerinde
çalıştıran bir sistem. Bu servislerden birine giren saldırgan iki tür iz bırakır:

| Katman | Veriyi toplayan araç | Ne görür |
|---|---|---|
| Ağ katmanı | Hubble (Cilium) | Hangi servis kime, hangi porta, hangi protokolle bağlandı |
| Çalışma zamanı katmanı | Tetragon | Hangi süreç doğdu, hangi dosya açıldı, hangi program çalıştırıldı |

İkisi de Linux çekirdeğindeki eBPF adlı gözlem mekanizmasını kullanıyor. Yani
uygulamanın koduna dokunmadan, çekirdek düzeyinde ölçüm yapabiliyoruz.

### Araştırma sorusu ve özgün katkı

İki katmanı ayrı ayrı izleyen çalışmalar var. Literatürde ikisini birlikte kullanan
çalışmalar ise katmanları **karar seviyesinde** birleştiriyor: her katman kendi alarmını
üretiyor, sonra bu alarmlar birleştiriliyor.

Tez bunu bir adım öteye taşıyor. İki katmanın ölçümlerini **feature seviyesinde**
birleştirip tek bir modele veriyoruz. Soru şu:

> İki katmanı feature seviyesinde birleştirmek, karar seviyesinde birleştirmekten daha
> iyi sonuç veriyor mu?

Bu soruyu cevaplamak için üç şey gerekiyor: iki katmanın verisi, her katmanın tek
başına çalışan birer **baseline** modeli ve birleşik model. Baseline modeller,
"birleştirmenin katkısı ne?" sorusunu ölçmenin yolu.

**Veriyi neden kendimiz topluyoruz?** Ağ akışları ile çalışma zamanı olaylarını aynı
kümede ve aynı zaman damgasıyla eşleştiren halka açık bir veri seti yok. Hazır veri
setlerini tek katmanlı doğrulama için kullanıyoruz; birleştirme için gereken veriyi
kendi ortamımızda üretiyoruz.

## 2 · Veri nasıl toplanıyor?

Ölçümün temel birimi **window**: sabit uzunlukta bir zaman dilimi (şu an 10 saniye).
Her window'da her servis için bir dizi sayı ölçüyoruz; bu sayıların her birine
**feature** diyoruz. Birkaç örnek:

- `out_flow_count` — bu servis bu window'da kaç ağ bağlantısı başlattı
- `distinct_dest_ports` — kaç farklı hedef porta bağlandı
- `exec_count` — kaç yeni süreç doğurdu
- `sensitive_file_count` — kaç hassas dosyaya erişti

Yani modelin gördüğü her satır, bir (servis, window) ikilisi ve onun feature vektörü.

**Label (etiket).** Modeli eğitmek için hangi window'un saldırı olduğunu bilmemiz
gerekiyor. Bunu kendimiz üretiyoruz: kümede bilinen saldırı senaryolarını çalıştırıp
zaman aralıklarını kaydediyoruz. Altı senaryo var:

`port-scan` · `dns-exfil` · `crypto-mining` · `unexpected-binary` ·
`sensitive-file-access` · `outbound-c2`

Saldırı aralığına denk gelmeyen window'lar `normal` etiketi alıyor. Aşağıdaki tablo,
iki katmanın hangi feature'ları ürettiğini gösteriyor.

In [2]:
layers = pd.DataFrame([
    {"katman": "ağ", "kaynak": "Hubble", "feature sayısı": len(NET_COLS),
     "feature'lar": ", ".join(NET_COLS)},
    {"katman": "çalışma zamanı", "kaynak": "Tetragon", "feature sayısı": len(RT_COLS),
     "feature'lar": ", ".join(RT_COLS)},
])
layers.set_index(["katman", "kaynak"])

,,feature sayısı,feature'lar
katman,kaynak,,
ağ,Hubble,5,"out_flow_count, out_drop_count, distinct_dest_..."
çalışma zamanı,Tetragon,6,"exec_count, exit_count, distinct_binaries, kpr..."


## 3 · Neredeyiz?

### Şu ana kadar ne yaptık?

- **Ortam.** Kümeye iki telemetri katmanını, metrik altyapısını ve izleyeceğimiz demo
  uygulamasını kurduk. Kurulum tek komutla tekrarlanabiliyor.
- **Literatür.** Araştırma boşluğunu yazdık: mevcut çalışmalar iki katmanı karar
  seviyesinde birleştiriyor, feature seviyesinde birleştiren bir çalışma yok.
- **Veri toplama.** İki katmanı aynı zaman aralığında kaydeden toplayıcıları yazdık;
  saldırı senaryolarını çalıştırıp etiketleri ürettik.
- **Feature çıkarımı.** Ham veriyi window bazında feature vektörlerine çevirdik.

Bu dört adımın çıktısı, bu notebook'ta inceleyeceğimiz veri seti.

### Yol haritası

```
Ortam → Literatür → Veri toplama → Feature çıkarımı
   → Veri kalitesi   (şu an buradayız)
   → Tek katman baseline → Birleşik model → Panel → Değerlendirme
```

Kritik nokta şu: sonraki bütün adımlar **tek bir veri setine** bağlı. Bu yüzden
sıradaki iş toplamayı tekrarlamak değil, veriyi toplama biçimini düzeltmek.

Aşağıdaki tablo faz durumlarını doğrudan yol haritasından okuyor.

In [ ]:
import re

#: Yol haritasındaki durum simgeleri kelimeye çevriliyor (kod noktasıyla yazıldı,
#: kaynakta simge görünmesin). Hücre yalnızca simgeden oluşuyorsa kelime kalıyor.
DURUM_SIMGE = {"✅": "tamamlandı", "\U0001F504": "sürüyor", "⏳": "başlamadı"}


def roadmap_phases(path):
    """Yol haritasındaki faz tablosunu okur (kopyalamaz)."""
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        cells = [c.strip() for c in line.strip().strip("|").split("|")]
        if len(cells) == 4 and cells[0][:1].isdigit():
            cells = [re.sub(r"\[([^\]]*)\]\([^)]*\)", r"\1", c) for c in cells]
            cells = [c.replace("**", "").replace("`", "").replace("★", "").strip()
                     for c in cells]
            durum = cells[1]
            for simge, kelime in DURUM_SIMGE.items():
                if simge in durum:
                    durum = durum.replace(simge, "").strip() or kelime
                    break
            rows.append({"faz": cells[0], "durum": durum})
    return pd.DataFrame(rows)


INDEX = REPO / "roadmap" / "README.md"
phases = roadmap_phases(INDEX)
if phases.empty:
    print(f"faz tablosu okunamadı: {INDEX}")
else:
    print(f"kaynak: {INDEX.relative_to(REPO)}")
phases.set_index("faz")

## 4 · Bulgu 1: veri hacmi yetmiyor

İlk toplama koşusunda saldırıları tek seferlik çalıştırıyorduk. Saldırı pod'u saniyeler
içinde işini bitirip kapanıyor, window'un geri kalanı boş kalıyordu. Sonuçta her senaryo
için yalnızca birkaç satır veri oluştu.

Bu hacimle şunlar yapılamaz:

- **Model eğitilemez.** Birkaç örnekle öğrenme olmaz.
- **Sonuç doğrulanamaz.** Test için ayrılacak veri kalmıyor.
- **Karşılaştırma yapılamaz.** Hangi katmanın ne katkı sağladığı ölçülemez.

Aşağıdaki grafikte her etiketin kaç window ürettiği görünüyor. Dikey işaretler kabul
eşikleri: `normal` için ve her senaryo için ayrı ayrı.

In [ ]:
fig, overview = charts.label_distribution(df, WINDOW)
plt.show()
overview.assign(durum=[style.status_text(w >= t)
                       for w, t in zip(overview["window sayısı"], overview["hedef"])])

## 5 · Bulgu 2: çalışma zamanı katmanı normalde sessiz

Bu, hacim sorunundan daha ciddi.

Demo uygulamamız uzun ömürlü servislerden oluşuyor. Bu servisler çalışırken yeni süreç
doğurmuyor, hassas dosya açmıyor. Bu yüzden `normal` window'larda çalışma zamanı
feature'ları neredeyse hep sıfır çıkıyor.

**Bu neden sorun?** Model iki sınıfı ayırt etmeyi şöyle öğreniyor: "bu window'da bir
süreç doğduysa saldırıdır." Bu kural normali ayırt ediyor gibi görünüyor, ama modele
normal davranış hakkında hiçbir şey öğretmiyor; çünkü ortada öğrenilecek bir normal
çalışma zamanı davranışı yok.

**Tezin iddiası için anlamı.** Birleştirmenin katkısı, ikinci katmanın bağımsız bilgi
taşımasına bağlı. O katman normal hakkında bilgi taşımıyorsa, ölçülecek bir katkı
kalmıyor.

Ölçüt olarak "normal window'ların en az bir çalışma zamanı feature'ı sıfırdan farklı
olma oranı"nı kullanıyoruz. Aşağıdaki grafik solda bu oranı, sağda feature bazında
dağılımı gösteriyor.

In [ ]:
fig, coverage = charts.coverage_panel(df)
plt.show()

## 6 · Bulgu 2'nin kanıtı: tek kurallı deneme

Bunu görmenin en kolay yolu hiç model eğitmemek. Tek bir kural deneyelim:

> "Bu window'da herhangi bir feature sıfırdan farklıysa, bunu saldırı say."

Bir kuralın işe yaraması için iki şey gerekiyor: saldırı window'larında tetiklenmeli,
normal window'larında tetiklenmemeli. Aşağıdaki tablo her feature için ikisini de
ölçüyor.

Karşılaştırma noktası **çoğunluk sınıfı**: hiçbir şey öğrenmeden "hep normal" demenin
doğruluğu. Bir kural bu değerin üzerine çıkmıyorsa hiçbir şeyi ayırmıyor demektir.

In [ ]:
actual = df.label != "normal"
baseline = float((~actual).mean())      # hiçbir şey öğrenmeden "hep normal" demek


def _num(frame, col):
    return pd.to_numeric(frame[col], errors="coerce").fillna(0)


rows = []
for c in ALL_COLS:
    fires = _num(df, c) > 0
    rows.append({
        "feature": c,
        "katman": "çalışma zamanı" if c in RT_COLS else "ağ",
        "kural doğruluğu": round(float((fires == actual).mean()), 3),
        "normalde tetiklenme": round(float(fires[~actual].mean()), 3),
        "saldırıda tetiklenme": round(float(fires[actual].mean()), 3),
    })
rules = pd.DataFrame(rows)
rules["çoğunluk sınıfına fark"] = (rules["kural doğruluğu"] - baseline).round(3)
rules = rules.sort_values("kural doğruluğu", ascending=False)
print(f"çoğunluk sınıfı (hep 'normal' demek): {baseline:.3f}")
rules

In [ ]:
# Aynı tablonun görseli: her feature için iki çubuk.
order = rules["feature"].tolist()
by_feature = rules.set_index("feature")
y = list(range(len(order)))[::-1]
fig, ax = plt.subplots(figsize=(8.6, 5.2))
ax.barh([yi + 0.2 for yi in y], [by_feature.loc[f, "normalde tetiklenme"] for f in order],
        height=0.36, color=style.S1, zorder=3, label="normal window'larda tetiklenme")
ax.barh([yi - 0.2 for yi in y], [by_feature.loc[f, "saldırıda tetiklenme"] for f in order],
        height=0.36, color=style.S2, zorder=3, label="saldırı window'larında tetiklenme")
ax.set_yticks(y)
ax.set_yticklabels(order)
ax.set_xlim(0, 1)
ax.set_xticks([0, .25, .5, .75, 1])
ax.set_xticklabels(["%0", "%25", "%50", "%75", "%100"])
ax.set_xlabel("kuralın tetiklenme oranı")
ax.xaxis.grid(True)
style.strip_frame(ax)
# Açıklama eksen dışında: çubukların çoğu tam genişliğe uzandığı için içeride her
# konum bir çubuğun üstüne biner.
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.01), ncol=2)
ax.set_title('Tek kurallı sınıflandırıcı: "feature > 0 ise saldırı"', pad=30)
plt.tight_layout()
plt.show()

In [ ]:
olu = rules[rules["saldırıda tetiklenme"] == 0]
temiz = rules[(rules["normalde tetiklenme"] == 0) & (rules["saldırıda tetiklenme"] > 0)]
fark = rules[rules["çoğunluk sınıfına fark"] > 0.02]

olu_ad = ", ".join(olu["feature"]) if len(olu) else "yok"
print(f"hiç tetiklenmeyen (ölü) feature                     : {len(olu)}/{len(rules)}  ({olu_ad})")
print(f"normalde hiç tetiklenmeyip saldırıda tetiklenen     : {len(temiz)}/{len(rules)}")
print(f"çoğunluk sınıfını 2 puandan fazla geçen kural       : {len(fark)}/{len(rules)}")
if len(temiz):
    print()
    print(temiz[["feature", "katman", "kural doğruluğu", "saldırıda tetiklenme"]].to_string(index=False))

## 7 · Bu sorun neden ilk bakışta görünmedi?

Ölçümleri özetlerken her senaryo için "onu normalden en iyi ayıran feature"ı ve bir
ayrılabilirlik skoru hesaplıyoruz. Bu skorlar ilk bakışta çok yüksek çıktı ve veri
sağlıklı göründü. Sebebi skorun tanımında:

> skor = |saldırı ortalaması − normal ortalaması| / normalin standart sapması

Paydada normalin kendi değişkenliği var. Normal window'larda çalışma zamanı aktivitesi
neredeyse hiç olmadığı için bu değer sıfıra çok yakın çıkıyor. Sıfıra yakın bir sayıya
bölmek, küçük farkları bile devasa skorlara dönüştürüyor. Yani yüksek skor "güçlü bir
sinyal bulundu" demek değil; "normalde öğrenilecek bir değişkenlik yok" anlamına
geliyor.

Aşağıdaki ısı haritasında satırlar senaryoları, sütunlar feature'ları gösteriyor. Koyu
hücreler yüksek skoru işaret ediyor. Bu koyuluğun sebebi güçlü bir tespit değil,
yukarıdaki payda.

In [ ]:
fig, scores, scenarios = charts.separability_heatmap(df)
plt.show()

## 8 · Plan: veriyi nasıl yeniden topluyoruz?

Dört değişiklik yapıyoruz.

**1. Normal trafiğe gerçek çalışma zamanı aktivitesi eklemek.** Kümeye sürekli çalışan
bir bakım iş yükü ekledik. Bu iş yükü düzenli aralıklarla süreç doğuruyor ve dosya
açıyor; böylece `normal` sınıfı da öğrenilecek bir çalışma zamanı davranışı kazanıyor.

*Not: bu iş yükünü biz ekledik, yapay. Tezde böyle belirtilecek. Gerçek kümelerde bu
tür aktivite yedekleme, log döndürme, zamanlanmış bakım gibi işlerden doğar.*

**2. Bozuk servisleri düzeltmek.** İki servis sürekli yeniden başlıyordu. Sebep
uygulamanın kendisi değil, sağlık kontrolü ayarıydı: servisin portu hazır olmadan
kontrol başarısız sayılıp konteyner öldürülüyordu. Başlangıç için 120 saniye tanıyınca
sorun ortadan kalktı.

**3. Etiketlemeyi düzeltmek.** Eski yöntem yalnızca saldırı pod'unun adına bakıyordu.
Bu yüzden gerçek bir servis üzerinden yapılan saldırı yanlışlıkla `normal` etiketleniyordu.
Yeni yöntem iki koşula birlikte bakıyor: window saldırı zaman aralığında olmalı ve pod
o saldırının hedefi olmalı.

**4. Saldırıları döngüye almak ve gerçek servislerin içinden çalıştırmak.** Saldırılar
artık ayrı ve geçici bir pod'da değil, çalışan bir servisin içinde koşuyor. Böylece
normal ve saldırı verisi aynı tür pod'dan geliyor; model "şu imaj saldırıdır" gibi bir
kısayol öğrenemiyor.

**Window boyutu.** Senaryolar 6 dakika sürdüğünde "senaryo başına en az 30 window"
kriteri 10 saniyelik window'da karşılanıyor, 30 saniyelikte karşılanmıyor. Bu yüzden
birincil window 10 saniye seçildi.

## 9 · Şu anki durum: kabul kriterleri

Verinin bir sonraki adıma geçmeye hazır olup olmadığını beş kriterle kontrol ediyoruz.
Aşağıdaki tablo, kullandığımız doğrulama betiğinin ürettiği tablonun aynısı.

| Ölçüt | Ne istiyoruz | Neden |
|---|---|---|
| Senaryo başına window | en az 30 | Bir senaryoyu öğrenmek ve ayrıca test edebilmek için |
| Normal window sayısı | en az 200 | Normal sınıfını temsil edebilmek için |
| Toplam satır | en az 500 | Eğitim ve test ayrımı yapabilmek için |
| Normalde çalışma zamanı kapsamı | en az %20 | Normal davranışın öğrenilebilmesi için |
| Ayırt edilemeyen senaryo | 0 | Her senaryo en az bir feature'la ayrılabilmeli |

Bu kriterler tutmuyorsa bir sonraki adıma geçmenin anlamı yok: tutmayan bir veriyle
eğitilen model yüksek doğruluk gösterebilir, ama bu doğruluk bir şey ifade etmez.

In [ ]:
undetectable = [s for s in scenarios if vd.best_discriminator(df, s) is None]

raw = pd.DataFrame(vd.check_targets(summary, coverage, undetectable),
                   columns=["kriter", "gerçek", "hedef", "tuttu"])
all_pass = bool(raw["tuttu"].all())
raw.assign(durum=[style.status_text(t) for t in raw["tuttu"]]).drop(columns=["tuttu"])

In [ ]:
print("SONUÇ: " + ("kabul edilebilir, bir sonraki adıma geçilebilir" if all_pass
                   else "henüz hazır değil, kriterler yukarıda"))
print(f"\nnot: bu tablo {'GÜNCEL' if not stale else 'ESKİMİŞ'} veriyle üretildi "
      f"({len(df)} satır, {features_zamani:%Y-%m-%d})")

## 10 · Sıradaki adım

Uzun ve temiz bir toplama koşusu yapıyoruz: uzun bir normal dönem, ardından altı
senaryonun döngüyle tekrarlandığı bir saldırı dönemi. Sonrasında feature'ları yeniden
üretip bu tabloyu tekrar okuyacağız.

Kabul kriterleri tutarsa bir sonraki faza geçilir. Tutmazsa hangi kriterin tutmadığı
tabloda göründüğü için doğrudan o kriter hedeflenir.

**Riskler.**

| Risk | Karşılık |
|---|---|
| Eklediğimiz bakım iş yükü yapay | Tezde açıkça belirtilecek |
| Ağ katmanının uygulama düzeyi (L7) görünürlüğü kapsam dışı kaldı | DNS sızıntısı senaryosu şu an yalnızca süreç doğurması sayesinde görünüyor. Bunu bir tespit boşluğu olarak yazacağız ve sonraki fazda kapatacağız |
| Bazı senaryolar zayıf sinyal taşıyor | Kabul kriteri her senaryonun ayrılabilir olmasını istiyor; olmayan senaryoyu ya güçlendireceğiz ya kapsamdan çıkaracağız |
| Ortam kısıtları | Tetragon bazı kuralları konteynere bağlayamıyor. Ölçülmüş kısıt olarak yöntem bölümüne girecek |

## Sözlük

| Terim | Anlamı |
|---|---|
| Kubernetes | Uygulamayı küçük servislere bölüp bir sunucu kümesinde çalıştıran sistem |
| Pod | Kubernetes'te çalışan en küçük birim; bir veya birkaç konteyner |
| eBPF | Uygulamayı değiştirmeden çekirdek düzeyinde gözlem yapmayı sağlayan Linux mekanizması |
| Hubble (Cilium) | Ağ akışlarını izleyen katman |
| Tetragon | Süreç ve dosya olaylarını izleyen katman |
| Window | Sabit uzunlukta zaman dilimi (şu an 10 saniye); ölçümün temel birimi |
| Feature | Bir window'da bir servis için hesaplanan sayısal ölçüm |
| Label | Bir window'un saldırı mı normal mi olduğunu söyleyen doğru bilgi |
| Senaryo | Simüle edilen tek bir saldırı türü |
| Normal | Saldırı içermeyen window'lar |
| Fusion | İki telemetri katmanını tek modelde birleştirme |
| Baseline | Karşılaştırma için kullanılan, tek katmanla çalışan basit model |
| Ayrılabilirlik | Bir feature'ın bir senaryoyu normalden ne kadar iyi ayırdığı |
| Standart sapma | Bir değer kümesinin ne kadar dağıldığının ölçüsü |
| L7 | Ağ katmanının uygulama düzeyi (HTTP, DNS gibi); şu an kapsam dışı |
